# Week 96, Day 2: Contribution sprint

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w096d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('walkdates.py', 'w', encoding='utf-8') as f:
    f.write('def is_leap(year):\n    if year % 400 == 0:\n        return True\n    if year % 100 == 0:\n        return False\n    return year % 4 == 0\n\n\ndef days_in_month(year, month):\n    if not 1 <= month <= 12:\n        raise ValueError(f"bad month: {month}")\n    if month == 2:\n        return 29 if is_leap(year) else 28\n    if month in (4, 6, 9, 11):\n        return 30\n    return 31\n\n\ndef week_number(day_of_year):\n    if day_of_year < 1:\n        raise ValueError("days count from 1")\n    return (day_of_year - 1) // 7 + 1\n')
with open('test_walkdates.py', 'w', encoding='utf-8') as f:
    f.write('import unittest\nfrom walkdates import days_in_month, is_leap\n\n\nclass TestWalkDates(unittest.TestCase):\n    def test_leap(self):\n        self.assertTrue(is_leap(2024))\n\n    def test_january(self):\n        self.assertEqual(days_in_month(2026, 1), 31)\n')

**Predict before you run:** what will this print?



In [ ]:
import sys

seen = []
def tracer(frame, event, arg):
    if frame.f_code.co_name == "sign":
        seen.append((event, frame.f_lineno - frame.f_code.co_firstlineno))
    return tracer

def sign(x):
    if x < 0:
        return -1
    return 1

sys.settrace(tracer)
sign(5)
sys.settrace(None)
print(seen)

> **Key idea.** Line coverage is the share of statements that run during the tests: it finds untested code but can't prove tests check anything. `sys.settrace` calls the tracer on `call`; its return value traces `line` and `return` events; filter by `co_filename` and restore the old tracer. Count statements from the syntax tree, subtract executed lines, and write a checking test for each gap.

### Exercise: Which lines ran?

Write `executed_lines(fn, *args)`: call `fn(*args)` with a tracer installed, and return the sorted line numbers (counted from the `def` line as 0, using `co_firstlineno`) of the lines that ran **in `fn` itself** (compare `frame.f_code` with `fn.__code__`). Restore the previous tracer afterwards, even if `fn` raises (and let the error propagate).

In [ ]:
import sys

def executed_lines(fn, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sys\n\ndef sign(x):\n    if x < 0:\n        return -1\n    return 1\n\ndef grade(score):\n    if score >= 70:\n        return \"pass\"\n    if score < 0:\n        raise ValueError(\"negative\")\n    return \"fail\"\n\ndef test_lines():\n    \"what ran\"\n    assert executed_lines(sign, 5) == [1, 3] and executed_lines(sign, -2) == [1, 2]\n    assert executed_lines(grade, 50) == [1, 3, 5]\n\ndef test_restores():\n    \"even when it raises\"\n    old = sys.gettrace()\n    try:\n        executed_lines(grade, -1)\n    except ValueError:\n        pass\n    else:\n        raise AssertionError(\"should raise\")\n    assert sys.gettrace() is old")

### Exercise: The statements to count

Write `statement_lines(source)`: the set of line numbers of every statement (`ast.stmt`) inside the module's functions, at any depth, leaving out each function's docstring.

In [ ]:
import ast

def statement_lines(source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_statements():\n    \"fifteen in walkdates\"\n    assert sorted(statement_lines(open(\"walkdates.py\").read())) == [2, 3, 4, 5, 6, 10, 11, 12, 13, 14, 15, 16, 20, 21, 22]\n    assert statement_lines('def f():\\n    \"\"\"Doc.\"\"\"\\n    return 1\\n\\nX = 2\\n') == {3}")

### Exercise: What no test runs

Using the given helpers, write `missing_by_function(source, filename, action)`: compile and execute `source` with that `filename` into a fresh namespace, then trace `action(namespace)` (recording lines from that file only), and return a list of `"NAME: lines A, B"` for each top-level function with statements that didn't run, in source order.

In [ ]:
import ast
import sys

def statement_lines(source):
    lines = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            body = node.body
            if ast.get_docstring(node) is not None:
                body = body[1:]
            for stmt in body:
                for sub in ast.walk(stmt):
                    if isinstance(sub, ast.stmt):
                        lines.add(sub.lineno)
    return lines

def traced(filename, action):
    hit = set()
    def tracer(frame, event, arg):
        if frame.f_code.co_filename != filename:
            return None
        if event == "line":
            hit.add(frame.f_lineno)
        return tracer
    old = sys.gettrace()
    sys.settrace(tracer)
    try:
        action()
    finally:
        sys.settrace(old)
    return hit

def missing_by_function(source, filename, action):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_existing_tests():\n    \"what the two existing tests reach\"\n    def tests(ns):\n        assert ns[\"is_leap\"](2024)\n        assert ns[\"days_in_month\"](2026, 1) == 31\n    assert missing_by_function(open(\"walkdates.py\").read(), \"<walkdates>\", tests) == [\n        \"is_leap: lines 3, 5\", \"days_in_month: lines 11, 13, 15\", \"week_number: lines 20, 21, 22\"]\n\ndef test_more():\n    \"fewer gaps\"\n    def tests(ns):\n        for year in (2000, 1900, 2024):\n            ns[\"is_leap\"](year)\n        ns[\"week_number\"](8)\n    assert missing_by_function(open(\"walkdates.py\").read(), \"<walkdates>\", tests) == [\n        \"days_in_month: lines 10, 11, 12, 13, 14, 15, 16\", \"week_number: lines 21\"]")

### Exercise: Filling the gaps

Write a `unittest` suite for `walkdates.py` that reaches every statement and catches all seven broken versions: centuries counted as leap years, no 400-year rule, February always 28 days, June with 31 days, month 13 accepted, weeks counted from 0, and day 0 accepted.

In [ ]:
import unittest
from walkdates import days_in_month, is_leap, week_number

class TestWalkDates(unittest.TestCase):
    def test_leap(self):
        self.assertTrue(is_leap(2024))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"walkdates.py\").read()\nBROKEN = {\n    \"centuries are leap\": GOOD.replace(\"    if year % 100 == 0:\\n        return False\", \"    if year % 100 == 0:\\n        return True\"),\n    \"no 400 rule\": GOOD.replace(\"    if year % 400 == 0:\\n        return True\", \"    if year % 400 == 0:\\n        return False\"),\n    \"February always 28\": GOOD.replace(\"return 29 if is_leap(year) else 28\", \"return 28\"),\n    \"June has 31\": GOOD.replace(\"(4, 6, 9, 11)\", \"(4, 9, 11)\"),\n    \"month 13 accepted\": GOOD.replace(\"1 <= month <= 12\", \"1 <= month <= 13\"),\n    \"weeks from 0\": GOOD.replace(\"// 7 + 1\", \"// 7\"),\n    \"day 0 accepted\": GOOD.replace(\"if day_of_year < 1:\", \"if day_of_year < 0:\"),\n}\n\ndef test_suite():\n    \"all seven caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"walkdates\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"walkdates.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: Coverage as a number

Write `coverage_line(total, missing)`: `"covered: C of T lines (P%)"` with C = T minus the number of missing lines and P rounded to a whole number, or `"covered: 0 of 0 lines (100%)"` when there are no statements.

In [ ]:
def coverage_line(total, missing):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_line():\n    \"a number, not a target\"\n    assert coverage_line(15, [3, 5, 11, 13, 15, 20, 21, 22]) == \"covered: 7 of 15 lines (47%)\"\n    assert coverage_line(15, []) == \"covered: 15 of 15 lines (100%)\" and coverage_line(0, []) == \"covered: 0 of 0 lines (100%)\"")

### Bug hunt: A tracer that saw nothing

The tracer returns `None` from the `call` event, which switches tracing off inside the function, so no `line` events arrive and this prints `[]`. Return the tracer so it prints `[1, 3]`.

In [ ]:
import sys

def sign(x):
    if x < 0:
        return -1
    return 1

hit = []
def tracer(frame, event, arg):
    if frame.f_code is sign.__code__ and event == "line":
        hit.append(frame.f_lineno - sign.__code__.co_firstlineno)

sys.settrace(tracer)
sign(5)
sys.settrace(None)
print(hit)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the local tracer\"\n    expect_output(\"[1, 3]\")")

### Bug hunt: Zero per cent

Integer division rounds 7 out of 15 down to `0` before it's multiplied, so this prints `0%`. Fix it so it prints `47%`.

In [ ]:
covered, total = 7, 15
print(f"{covered // total * 100}%")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"true division, then round\"\n    expect_output(\"47%\")")

### Puzzle: A tracer for one file

Arrange a tracer that records lines from one file and ignores everything else. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the three events a local tracer sees in a simple function, in order of first appearance, as a list
q = ___   # the year that reaches "return True" for the 400-year rule (any one)
r = ___   # True or False: 100% line coverage proves the tests check every result
s = ___   # the function that returns the tracer currently installed (module.function)
t = ___   # the free tool used for coverage on real projects (its PyPI name)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"call\", \"line\", \"return\"] and q % 400 == 0 and r is False, \"p, q and r\"\n    assert s == \"sys.gettrace\" and t == \"coverage\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a coverage report

Write `coverage_report(module_source, test_source, module_name)` with the given helpers. Compile the module with the file name `<MODULE_NAME>`, execute it into a new `types.ModuleType`, put it in `sys.modules` (restoring the old entry afterwards), execute `test_source`, and run its `unittest.TestCase` classes while tracing the module's file. Return lines: `"tests: N, failing: F"`, the coverage line (`covered: C of T lines (P%)`), and `"  NAME: lines A, B"` for each function with untested statements.

In [ ]:
import ast
import sys
import types
import unittest

def statement_lines(source):
    lines = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            body = node.body
            if ast.get_docstring(node) is not None:
                body = body[1:]
            for stmt in body:
                for sub in ast.walk(stmt):
                    if isinstance(sub, ast.stmt):
                        lines.add(sub.lineno)
    return lines

def traced(filename, action):
    hit = set()
    def tracer(frame, event, arg):
        if frame.f_code.co_filename != filename:
            return None
        if event == "line":
            hit.add(frame.f_lineno)
        return tracer
    old = sys.gettrace()
    sys.settrace(tracer)
    try:
        action()
    finally:
        sys.settrace(old)
    return hit

def coverage_report(module_source, test_source, module_name):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_existing():\n    \"the tests as they were\"\n    assert coverage_report(open(\"walkdates.py\").read(), open(\"test_walkdates.py\").read(), \"walkdates\") == [\n        \"tests: 2, failing: 0\", \"covered: 7 of 15 lines (47%)\", \"  is_leap: lines 3, 5\",\n        \"  days_in_month: lines 11, 13, 15\", \"  week_number: lines 20, 21, 22\"]\n\ndef test_more():\n    \"after the sprint\"\n    more = open(\"test_walkdates.py\").read() + '''\n    def test_everything(self):\n        from walkdates import week_number\n        for y in (2000, 1900):\n            is_leap(y)\n        for m in (2, 4):\n            days_in_month(2026, m)\n        week_number(3)\n        for bad in ((days_in_month, 2026, 13), (week_number, 0)):\n            with self.assertRaises(ValueError):\n                bad[0](*bad[1:])\n'''\n    assert coverage_report(open(\"walkdates.py\").read(), more, \"walkdates\") == [\"tests: 3, failing: 0\", \"covered: 15 of 15 lines (100%)\"]\n    import sys\n    assert sys.modules.get(\"walkdates\") is None or sys.modules[\"walkdates\"].__name__ == \"walkdates\"")

## Warm-up

Python called the tracer when `sign` started (`call`), for each line it ran (`line`), and when it finished (`return`). Line 2 of `sign`, `return -1`, never ran: a test calling `sign(5)` alone leaves it untested. That's line coverage in miniature.

## What coverage measures

**Line coverage** is the fraction of a module's statements that run while its tests run. `walkdates.py` and its test file `test_walkdates.py` are on this page: two tests, and more than half the statements never run. Coverage finds code **no test exercises**; it can't prove the tests are good (a line can run without anything checking its result, which is why Week 95's mutants matter too). Use it to find where to look, not as a score to chase. On your own computer, `coverage.py` (free) does this for real projects: `python -m coverage run -m pytest` then `python -m coverage report -m`; CPython has its own instructions in the devguide.

## Tracing

`sys.settrace(tracer)` asks Python to call `tracer(frame, event, arg)` whenever a function starts (`"call"`); whatever the tracer returns becomes the **local** tracer for that function, called for each `"line"` and for `"return"`. Return the tracer itself to keep tracing, or `None` to ignore that function. `frame.f_code.co_filename` tells you which file the code came from, so a coverage tool can record only the module it measures; `frame.f_lineno` is the line about to run. Always restore the previous tracer afterwards (a debugger may be using it).

The **statements** to count come from the syntax tree: every statement inside the module's functions, without docstrings. Compare them with the executed lines, and what's left is untested.

## Filling the gaps

For each untested line, ask which input reaches it: line 3 of `walkdates.py` (`return True` for years divisible by 400) needs 2000; line 5 needs 1900; the `ValueError` lines need a bad month or day. Then write the test so it **checks** the result, and each test for one behaviour. Coverage tells you you've reached the line; a mutant you kill tells you you've tested it.

## Practice

## Quiz

## Daily challenge

## Your sprint

1. **Measure coverage** of a small module in a project you use (`python -m coverage run -m pytest`, then `coverage report -m`), or of a standard library module with the devguide's instructions.
2. **Pick the gaps** that matter: error paths and branches, not trivial lines.
3. **Write checking tests** for them, in the project's style, and check each would catch a mistake.
4. **Open the pull request** as test-only, and log it.

## Before you go

Tomorrow's sprint is a bug fix from start to finish, at speed.